# datafake
### A Python library for generating realistic synthetic datasets

> **datafake** generates plausible, reproducible, and customizable synthetic datasets across 10 different domains. It uses real statistical distributions (Poisson, lognormal, Dirichlet, exponential) and enforces logical consistency between columns — making it ideal for testing pipelines, building dashboards, teaching data analysis, and running experiments without relying on real or sensitive data.

---

## Table of Contents
1. [Installation](#installation)
2. [The 10 Built-in Datasets](#datasets)
3. [Advanced Parameters](#advanced)
4. [Relational Datasets](#relational)
5. [Custom Datasets](#custom)
6. [Dataset Summary](#describe)
7. [Common Mistakes](#mistakes)


---
## 1. Installation

Install `datafake` directly from PyPI. All dependencies (`pandas`, `numpy`, `faker`) are installed automatically.

In [ ]:
!pip install datafake

Once installed, import the functions you need:

In [ ]:
from datafake import (
    generate_sales, generate_users, generate_football,
    generate_music, generate_weather, generate_social,
    generate_movies, generate_health, generate_flights,
    generate_elections, generate_products,
    generate_related, generate_custom, describe_dataset
)

print('datafake imported successfully')
print('Available generators: sales, users, football, music, weather,')
print('                      social, movies, health, flights, elections,')
print('                      products, related, custom')

---
## 2. The 10 Built-in Datasets

Each generator returns a `pandas.DataFrame` ready to use. All parameters are optional — calling a function with no arguments uses sensible defaults.

| Generator | Domain | Default rows |
|---|---|---|
| `generate_sales()` | E-commerce transactions | 500 |
| `generate_users()` | App users & churn | 300 |
| `generate_football()` | Football match stats | 500 |
| `generate_music()` | Music streaming | 500 |
| `generate_weather()` | Climate records | 500 |
| `generate_social()` | Social media posts | 500 |
| `generate_movies()` | Streaming movies | 500 |
| `generate_health()` | Patient records | 500 |
| `generate_flights()` | Flight data | 500 |
| `generate_elections()` | Fictional elections | 500 |

### Sales
Simulates e-commerce transactions with dates, products, amounts, and payment methods. Revenue is derived as `quantity × unit_price`, ensuring logical consistency.

In [ ]:
df_sales = generate_sales(n=500, seed=42, start_date='2024-01-01', end_date='2024-12-31')
print(df_sales.shape)
df_sales.head()

### Users
Generates a user base with subscription plans, devices, session activity, and churn status. The `churned` column is derived directly from `status`, not generated independently.

In [ ]:
df_users = generate_users()  # using all defaults
print(df_users.shape)
df_users.head()

### Football
Simulates football matches using a **Poisson distribution** for goals — the standard statistical model for scoring events. Home teams have a higher lambda (1.5 vs 1.1) to reflect the real home advantage effect. Results are derived from goals, not generated independently.

In [ ]:
df_football = generate_football(n=500, seed=7)
print(df_football.shape)
df_football.head()

### Music
Generates a music streaming catalog. Streams and likes use a **lognormal distribution** because music virality is highly skewed — most songs have few plays, while a few have billions.

In [ ]:
df_music = generate_music()
print(df_music.shape)
df_music.head()

### Weather
Generates climate records across 10 global cities. Temperature uses a **normal distribution**, precipitation uses an **exponential distribution** (most days have little rain, occasionally a lot), and wind speed uses a **lognormal distribution**.

In [ ]:
df_weather = generate_weather(n=500, seed=42, start_date='2024-01-01', end_date='2024-12-31')
print(df_weather.shape)
df_weather.head()

### Social Media
Simulates social media posts with engagement metrics. All interaction columns (likes, comments, shares, reach) use **lognormal distributions** because social media engagement is extremely skewed. The `engagement_rate` is derived from existing columns.

In [ ]:
df_social = generate_social()
print(df_social.shape)
df_social.head()

### Movies
Generates a streaming movie catalog. IMDb scores use a **normal distribution** centered at 6.5, clipped to the valid [1, 10] range. Box office uses **lognormal** because most movies earn little and a few earn billions.

In [ ]:
df_movies = generate_movies(n=500, seed=42)
print(df_movies.shape)
df_movies.head()

### Health
Generates synthetic patient records. Weight and height use **normal distributions** with clipping to avoid physiologically impossible values. BMI is derived using the official medical formula: `weight_kg / (height_cm / 100) ** 2`.

In [ ]:
df_health = generate_health()
print(df_health.shape)
df_health.head()

### Flights
Simulates flight data with airlines, routes, prices, and delays. Ticket prices use **lognormal** distribution. `delay_min` is only non-zero when `status == 'delayed'`, enforcing logical consistency between columns.

In [ ]:
df_flights = generate_flights(n=500, seed=42)
print(df_flights.shape)
df_flights.head()

### Elections
Generates fictional election results. Vote distribution uses a **Dirichlet distribution** — the statistically correct way to distribute proportions across multiple candidates that must sum to 100%. `vote_share_pct` is derived from `votes / total_votes_region`.

In [ ]:
df_elections = generate_elections()
print(df_elections.shape)
df_elections.head()

---
## 3. Advanced Parameters

All generators share a common set of optional parameters that control reproducibility, data quality simulation, language, and output format.

| Parameter | Type | Default | Description |
|---|---|---|---|
| `n` | int | varies | Number of rows |
| `seed` | int | 42 | Random seed for reproducibility |
| `missing_rate` | float | 0.0 | Proportion of NaN values to inject |
| `noise_level` | float | 0.0 | Gaussian noise + outliers on numeric columns |
| `locale` | str | `'en_US'` | Faker locale for text generation |
| `save_to` | str | None | File path to export (.csv or .xlsx) |

### missing_rate
Injects NaN values randomly across non-protected columns. ID columns and date columns are always protected and will never receive NaNs.

In [ ]:
df_with_nans = generate_sales(n=500, seed=42, missing_rate=0.2)

print('NaN count per column:')
print(df_with_nans.isnull().sum())
print()
print('Notice: sale_id, date, customer_id, product_id have 0 NaNs — they are protected.')

### noise_level
Adds Gaussian perturbations to numeric columns and injects extreme outliers (~2% of values). This makes datasets more realistic for testing pipelines that need to handle messy data.

In [ ]:
df_clean = generate_sales(n=500, seed=42)
df_noisy = generate_sales(n=500, seed=42, noise_level=0.15)

print('=== Without noise ===')
print(df_clean['unit_price'].describe().round(2))
print()
print('=== With 15% noise ===')
print(df_noisy['unit_price'].describe().round(2))
print()
print('Notice: std and max are higher with noise — outliers were injected.')

### locale
Changes the language used by Faker for names, cities, and emails. Useful for generating region-specific datasets.

In [ ]:
df_en = generate_users(n=5, seed=42, locale='en_US')
df_es = generate_users(n=5, seed=42, locale='es_MX')
df_fr = generate_users(n=5, seed=42, locale='fr_FR')

print('=== English (en_US) ===')
print(df_en[['name', 'email', 'country']])
print()
print('=== Spanish Mexico (es_MX) ===')
print(df_es[['name', 'email', 'country']])
print()
print('=== French (fr_FR) ===')
print(df_fr[['name', 'email', 'country']])

### save_to
Exports the DataFrame directly to a file. Supports `.csv` and `.xlsx` formats. The DataFrame is still returned normally so you can continue using it.

In [ ]:
df = generate_health(n=200, seed=42, save_to='health_data.csv')
df2 = generate_movies(n=200, seed=42, save_to='movies_data.xlsx')
print('Both files saved. DataFrames are still available for use:')
print(df.shape, df2.shape)

### Reproducibility
Using the same `seed` always produces the exact same dataset. This is critical for testing, teaching, and scientific experimentation.

In [ ]:
df1 = generate_sales(seed=99)
df2 = generate_sales(seed=99)
df3 = generate_sales(seed=0)  # different seed

print(f'Same seed (99 vs 99): {df1.equals(df2)}')
print(f'Different seed (99 vs 0): {df1.equals(df3)}')

---
## 4. Relational Datasets

`generate_related()` generates three interconnected tables — customers, products, and sales — with **consistent IDs** across all of them. This means you can perform correct joins, just like you would with a real relational database.

This is particularly useful for:
- Testing ETL pipelines
- Building dashboards with multiple data sources
- Practicing SQL joins and aggregations

In [ ]:
data = generate_related(
    n_customers=200,
    n_products=50,
    n_sales=1000,
    seed=42
)

customers = data['customers']
products  = data['products']
sales     = data['sales']

print('Table shapes:')
print(f'  customers: {customers.shape}')
print(f'  products:  {products.shape}')
print(f'  sales:     {sales.shape}')

print()
print('Sample from each table:')
print(customers.head(2))
print(products.head(2))
print(sales.head(2))

### Joining the tables
Because IDs are consistent, merges work correctly — every `customer_id` in sales exists in customers, and every `product_id` in sales exists in products.

In [ ]:
merged = (
    sales
    .merge(customers, on='customer_id')
    .merge(products, on='product_id', suffixes=('_customer', '_product'))
)

print(f'Merged table: {merged.shape}')
merged[['sale_id', 'name', 'segment', 'name_product', 'category', 'revenue']].head()

### Analysis example — revenue by customer segment

In [ ]:
import matplotlib.pyplot as plt

revenue_by_segment = merged.groupby('segment')['revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(revenue_by_segment.index, revenue_by_segment.values,
               color=['steelblue', 'tomato', 'seagreen'], edgecolor='white')
plt.title('Total Revenue by Customer Segment')
plt.xlabel('Segment')
plt.ylabel('Revenue (USD)')
for bar, val in zip(bars, revenue_by_segment.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'${val:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### Saving all related tables at once
Use `save_to` to export all three tables to a folder automatically.

In [ ]:
data = generate_related(
    n_customers=200,
    n_products=50,
    n_sales=1000,
    seed=42,
    save_to='related_output'
)
# This creates: related_output/customers.csv, products.csv, sales.csv

---
## 5. Custom Datasets

`generate_custom()` lets you define your own schema and datafake will generate the data automatically. You define column names, types, and parameters — datafake handles the rest.

**Supported types:**

| Type | Description | Parameters |
|---|---|---|
| `int` | Integer values | `min`, `max` |
| `float` | Float values | `min`, `max` |
| `normal` | Normal distribution | `mean`, `sigma` |
| `lognormal` | Lognormal distribution | `mean`, `sigma` |
| `bool` | Boolean values | `p` (probability of True) |
| `category` | Categorical values | `values`, `weights` |
| `name` | Fake full names | — |
| `email` | Fake email addresses | — |
| `city` | Fake city names | — |
| `country` | Fake country names | — |
| `date` | Random dates | `start`, `end` |
| `text` | Random sentences | — |

In [ ]:
schema = {
    'name':           {'type': 'name'},
    'age':            {'type': 'int', 'min': 18, 'max': 65},
    'salary':         {'type': 'lognormal', 'mean': 10, 'sigma': 0.5},
    'city':           {'type': 'city'},
    'active':         {'type': 'bool', 'p': 0.8},
    'level':          {'type': 'category', 'values': ['Junior', 'Mid', 'Senior'], 'weights': [0.4, 0.4, 0.2]},
    'start_date':     {'type': 'date', 'start': '-3y', 'end': 'today'},
    'score':          {'type': 'normal', 'mean': 75, 'sigma': 10},
    'notes':          {'type': 'text'},
}

df_custom = generate_custom(schema, n=500, seed=42)
print(df_custom.shape)
df_custom.head()

### Custom dataset with all advanced parameters
`generate_custom()` also supports `missing_rate`, `noise_level`, `locale`, and `save_to`.

In [ ]:
df_custom_advanced = generate_custom(
    schema,
    n=500,
    seed=42,
    locale='es_MX',
    missing_rate=0.1,
    noise_level=0.05,
    save_to='custom_output.csv'
)

print('NaN count per column:')
print(df_custom_advanced.isnull().sum())

---
## 6. Dataset Summary

`describe_dataset()` generates a detailed per-column summary of any datafake DataFrame. It shows data types, null counts, unique values, and statistics for numeric columns or top frequent values for categorical ones.

In [ ]:
df_health = generate_health(n=500, seed=42, missing_rate=0.1)
summary = describe_dataset(df_health)
summary

In [ ]:
# describe_dataset works on any DataFrame, not just datafake ones
summary_sales = describe_dataset(generate_sales(missing_rate=0.15))
summary_sales

---
## 7. Common Mistakes

Here are the most frequent errors when using `datafake` and how to fix them.

### Mistake 1 — missing_rate out of range
`missing_rate` must be strictly between 0.0 and 1.0. Values of exactly 0.0 or 1.0 are not valid.

In [ ]:
# WRONG — will raise ValueError
try:
    df = generate_sales(missing_rate=1.5)
except ValueError as e:
    print(f'Error: {e}')

# CORRECT
df = generate_sales(missing_rate=0.2)
print('Works correctly:', df.shape)

### Mistake 2 — noise_level out of range
Same rule applies to `noise_level` — must be between 0.0 and 1.0 (exclusive).

In [ ]:
# WRONG
try:
    df = generate_sales(noise_level=2.0)
except ValueError as e:
    print(f'Error: {e}')

# CORRECT
df = generate_sales(noise_level=0.1)
print('Works correctly:', df.shape)

### Mistake 3 — wrong file extension in save_to
`save_to` only supports `.csv` and `.xlsx`. Any other extension raises an error.

In [ ]:
# WRONG
try:
    df = generate_sales(save_to='output.json')
except ValueError as e:
    print(f'Error: {e}')

# CORRECT
df = generate_sales(save_to='output.csv')
print('Saved to CSV correctly')

df = generate_sales(save_to='output.xlsx')
print('Saved to Excel correctly')

### Mistake 4 — unsupported type in generate_custom
Only the 12 supported types are valid in a custom schema. Using an unknown type raises a clear error.

In [ ]:
# WRONG — 'phone' is not a supported type
try:
    df = generate_custom({'phone': {'type': 'phone'}}, n=10)
except ValueError as e:
    print(f'Error: {e}')

# CORRECT — use 'text' or 'int' as alternatives
df = generate_custom({'phone': {'type': 'int', 'min': 1000000, 'max': 9999999}}, n=5)
print(df)

### Mistake 5 — category weights don't sum to 1
If `weights` are provided for a `category` type, they don't need to sum exactly to 1 — datafake normalizes them automatically. However, all weights must be positive.

In [ ]:
# This works — weights are automatically normalized
schema = {
    'tier': {'type': 'category', 'values': ['A', 'B', 'C'], 'weights': [3, 2, 1]}
}
df = generate_custom(schema, n=1000, seed=42)
print('Actual proportions (should be ~50%, ~33%, ~17%):')
print(df['tier'].value_counts(normalize=True).round(2))

### Mistake 6 — expecting different results with the same seed
The seed guarantees reproducibility. If you want different data, use a different seed.

In [ ]:
df_a = generate_sales(n=5, seed=42)
df_b = generate_sales(n=5, seed=42)  # identical
df_c = generate_sales(n=5, seed=123) # different

print('df_a == df_b:', df_a.equals(df_b))  # True
print('df_a == df_c:', df_a.equals(df_c))  # False
print()
print('To get varied data, change the seed:')
for s in [1, 2, 3]:
    df = generate_sales(n=3, seed=s)
    print(f'  seed={s}: first revenue = {df["revenue"].iloc[0]}')